In [7]:
"""
07b - Block-wise inference over the full region of interest at 30 m

Applies the trained model to every eligible pixel in the ROI, block by block, and
writes five rasters:

    severity_30m.tif          persistence_30m.tif      recovery_gap_30m.tif
    burned_prob_30m.tif       model_recurrence_30m.tif

model_recurrence is the count of months in which the classification head exceeds the
F1-optimal operating threshold of 0.502, normalised by 5 and capped at 1. That is the
same definition the analytical recurrence target uses at training time, so it is the
model-derived counterpart to recurrence_30m.tif, which is built from the
fire-reference inventory in the final cells.

The model was trained with a masked multi-task loss, so the regression heads received
gradient only from pixels with at least one documented fire month. At inference the
regression outputs are emitted everywhere and burned_prob is emitted alongside them,
so downstream analysis can condition the indicators on the burn mask rather than
assuming the regression values are meaningful off it.
"""

'\nNotebook 07b v4 — Block-wise inference with model-derived recurrence\n\nThis is v4 of the inference notebook. In addition to the four outputs\nfrom v3 (severity, persistence, recovery_gap, burned_prob), it also\nproduces a fifth raster: model_recurrence_30m.tif.\n\nmodel_recurrence is computed per pixel as the count of months in\nwhich the classification head produced a burned probability above\nthe F1-optimized threshold (0.502), normalized by 5 and capped at 1.\nThis matches the analytical recurrence definition used at training\ntime and gives a model-derived counterpart to the ground-truth-derived\nrecurrence raster produced from the fire polygons in Cell 21.\n\nAll other outputs (severity, persistence, recovery_gap, burned_prob)\nare identical to v3 and unchanged.\n'

# 07b - Block-wise full-ROI inference at 30 m

**Purpose** - run the trained model over the entire region of interest and write the
four indicator rasters plus the classification probability and the model-derived
recurrence raster.

**Main inputs** - the monthly feature rasters exported by notebook 07a, the trained
model and scaler from notebook 05, and the eligibility mask.

**Main outputs** - `outputs/inference_full_roi/{severity,persistence,recovery_gap,
burned_prob,model_recurrence,recurrence}_30m.tif`.

**Role in the thesis workflow** - the supplementary 30 m regional inference. These
rasters are read by notebook 08 section 16, by the Section 4.12 tables and by the
appendix figures. The final regional analysis in the thesis is the 10 m pipeline in
`pipeline_10m/`.

In [8]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/thesis")
import config
importlib.reload(config)
import model_defs
importlib.reload(model_defs)
config.ensure_dirs()

In [10]:
import re
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available — using CPU.")

Running on: cuda
GPU: Tesla T4


## 1. Locate the exported feature rasters

Some months are absent from the S2 collection (e.g. very cloudy
periods) and were skipped by Notebook 07a. We build a mapping from
canonical m_idx (0..107) to file path; missing m_idx values will be
NaN-padded in the inference tensor.

In [11]:
# Auto-detect environment (works both Colab and local)
import sys, os
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    FEATURE_DIR = Path("/content/drive/MyDrive/thesisv4/data/raw/feature_stack_v4_30m")
else:
    FEATURE_DIR = Path(THESIS_ROOT) / "data" / "raw" / "feature_stack_v4_30m"
assert FEATURE_DIR.exists(), f"Feature folder not found: {FEATURE_DIR}"

feature_files_all = sorted(FEATURE_DIR.glob("features_m*.tif"))
print(f"Found {len(feature_files_all)} feature rasters")

m_idx_to_file = {}
for f in feature_files_all:
    match = re.search(r"features_m(\d+)\.tif", f.name)
    if match:
        m_idx_to_file[int(match.group(1))] = f

T = len(config.get_month_list())
present_m_idx = sorted(m_idx_to_file.keys())
missing_m_idx = [i for i in range(T) if i not in m_idx_to_file]

print(f"T (canonical):    {T}")
print(f"Present m_idx:    {len(present_m_idx)} "
      f"(range {present_m_idx[0]}..{present_m_idx[-1]})")
print(f"Missing m_idx:    {missing_m_idx}")

Found 108 feature rasters
T (canonical):    108
Present m_idx:    108 (range 0..107)
Missing m_idx:    []


## 2. Copy feature rasters to local disk

Reading 106 files through the Drive FUSE mount causes two problems:
(1) FUSE caches every read, so the local disk fills to ~22 GB even
though we only read small windows; (2) FUSE occasionally drops the
connection ("Transport endpoint is not connected") when too many files
are held open concurrently. Copying the rasters to local disk once
eliminates both. The Colab VM has ~100 GB of local disk, easily
sufficient for the ~22 GB feature stack.

In [12]:
import shutil, os

# On local runs, feature_stack is already on the fast disk - skip the copy.
if IS_COLAB:
    LOCAL_FEATURE_DIR = Path("/content/feature_stack_local")
else:
    LOCAL_FEATURE_DIR = FEATURE_DIR  # already local, no copy needed
LOCAL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

local_m_idx_to_file = {}
copy_count = 0
for m_idx, drive_path in sorted(m_idx_to_file.items()):
    local_path = LOCAL_FEATURE_DIR / drive_path.name
    # Only copy if not already present at the correct size
    if not local_path.exists() or \
            local_path.stat().st_size != drive_path.stat().st_size:
        shutil.copy2(drive_path, local_path)
        copy_count += 1
    local_m_idx_to_file[m_idx] = local_path

total_size_gb = sum(f.stat().st_size for f in LOCAL_FEATURE_DIR.glob("*.tif")) / (1024**3)
print(f"Copied {copy_count} rasters to local disk "
      f"({len(local_m_idx_to_file)} total, {total_size_gb:.1f} GB)")

# Use the local paths from here on
m_idx_to_file = local_m_idx_to_file

Copied 107 rasters to local disk (108 total, 35.7 GB)


## 3. Open all raster handles and read spatial metadata

In [13]:
raster_handles = {m_idx: rasterio.open(path)
                  for m_idx, path in m_idx_to_file.items()}
print(f"Opened {len(raster_handles)} raster handles.")

first_handle = raster_handles[present_m_idx[0]]
raster_meta      = first_handle.meta.copy()
H, W             = first_handle.height, first_handle.width
raster_crs       = first_handle.crs
raster_transform = first_handle.transform
n_bands          = first_handle.count

print(f"Raster shape: {H} × {W} pixels")
print(f"Bands:        {n_bands}")
print(f"CRS:          {raster_crs}")

GEE_FEATURES = [
    f for f in config.MODEL_FEATURES
    if f not in config.DELTA_FEATURES and f not in config.HARMONIC_FEATURES
]
assert n_bands == len(GEE_FEATURES), \
    f"Expected {len(GEE_FEATURES)} bands in the raster, got {n_bands}"
print(f"Raster band count matches GEE_FEATURES ({n_bands}); "
      f"delta and harmonic features will be computed at runtime "
      f"to reach {len(config.MODEL_FEATURES)} total features.")

Opened 108 raster handles.
Raster shape: 2598 × 1625 pixels
Bands:        46
CRS:          EPSG:32636
Raster band count matches GEE_FEATURES (46); delta and harmonic features will be computed at runtime to reach 52 total features.


## 3b. Load the eligibility mask at 30 m

In [14]:
ECO_MASK_10M = config.FILES["burn_count_tif"]

with rasterio.open(ECO_MASK_10M) as src:
    eco_mask_30m = src.read(
        2,                                # band 2 = eco mask
        out_shape=(H, W),
        resampling=Resampling.mode,       # most common value per window
    )

eligible_count = (eco_mask_30m == 1).sum()
print(f"Eco mask resampled to {H} × {W}")
print(f"Eligible pixels: {eligible_count:,} / {eco_mask_30m.size:,} "
      f"({100 * eligible_count / eco_mask_30m.size:.1f}%)")

Eco mask resampled to 2598 × 1625
Eligible pixels: 1,827,777 / 4,221,750 (43.3%)


## 4. Load the trained model and the scaler

In [15]:
from model_defs import FireRecoveryModel

scaler_path = config.FILES["scaler"]
with open(scaler_path, "rb") as f:
    scaler_bundle = pickle.load(f)

scaler        = scaler_bundle["scaler"]
spectral_cols = scaler_bundle["spectral_cols"]
lc_cols       = scaler_bundle["lc_cols"]
print(f"Scaler loaded. Spectral cols: {len(spectral_cols)}, "
      f"LC cols: {len(lc_cols)}")

model = FireRecoveryModel(
    n_features = len(config.MODEL_FEATURES),
    d_model    = config.MODEL["d_model"],
    n_head     = config.MODEL["n_head"],
    d_k        = config.MODEL["d_k"],
    max_len    = T,
    tau        = config.MODEL["positional_tau"],
    dropout    = 0.0,
).to(DEVICE)

state = torch.load(config.FILES["model_main"], map_location=DEVICE)
model.load_state_dict(state)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Scaler loaded. Spectral cols: 32, LC cols: 20
Model loaded: 66,436 parameters


## 5. Output rasters

In [16]:
if IS_COLAB:
    OUT_DIR = Path("/content/drive/MyDrive/thesisv4/outputs/inference_full_roi")
else:
    OUT_DIR = Path(THESIS_ROOT) / "outputs" / "inference_full_roi"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_meta = raster_meta.copy()
out_meta.update(count=1, dtype="float32", nodata=np.nan)

out_paths = {
    "severity":         OUT_DIR / "severity_30m.tif",
    "persistence":      OUT_DIR / "persistence_30m.tif",
    "recovery_gap":     OUT_DIR / "recovery_gap_30m.tif",
    "burned_prob":      OUT_DIR / "burned_prob_30m.tif",
    "model_recurrence": OUT_DIR / "model_recurrence_30m.tif",
}

out_handles = {name: rasterio.open(path, "w", **out_meta)
               for name, path in out_paths.items()}

print(f"Output rasters will be written to: {OUT_DIR}")

Output rasters will be written to: /content/drive/MyDrive/thesisv4/outputs/inference_full_roi


## 6. Block-wise inference

The model was trained with `MaskedMultiTaskLoss`: the regression heads receive
gradient only from pixels with at least one documented fire month. At inference the
regression predictions are emitted for every pixel and the burned classification
probability is emitted separately, so downstream analysis can display the indicators
conditional on `burned_prob` crossing the operating threshold rather than treating
them as valid everywhere.

In [17]:
BLOCK_SIZE      = 256   # 256x256 blocks (was 512) to fit in Colab's ~13 GB RAM
INFERENCE_BATCH = 2048 if DEVICE == "cpu" else 4096

print(f"Block size:      {BLOCK_SIZE} × {BLOCK_SIZE}")
print(f"Inference batch: {INFERENCE_BATCH}")


def iter_blocks(H, W, block_size):
    for r in range(0, H, block_size):
        for c in range(0, W, block_size):
            yield r, c, min(block_size, H - r), min(block_size, W - c)


all_blocks = list(iter_blocks(H, W, BLOCK_SIZE))
print(f"Total blocks: {len(all_blocks)}")

Block size:      256 × 256
Inference batch: 4096
Total blocks: 77


In [18]:
print("Starting block-wise inference...")
start_time = time.time()

n_features = len(config.MODEL_FEATURES)
n_gee_features = len(GEE_FEATURES)
spectral_cols_arr = np.array(spectral_cols, dtype=int)
lc_cols_arr       = np.array(lc_cols,       dtype=int)

# Precompute indices for the delta source bands within the GEE-exported
# tensor, and the target indices for delta features in the full 52-feature
# tensor. Also precompute the harmonic features once for every calendar month.
delta_source_indices = {
    b: GEE_FEATURES.index(b) for b in config.DELTA_SOURCE_INDICES
}
delta_target_indices = {
    f"delta_{b}": config.MODEL_FEATURES.index(f"delta_{b}")
    for b in config.DELTA_SOURCE_INDICES
}
harmonic_target_indices = {
    name: config.MODEL_FEATURES.index(name)
    for name in config.HARMONIC_FEATURES
}
# For canonical timeline, calendar_months[t] = calendar month of time index t
month_list = config.get_month_list()
calendar_months = np.array([m for _, m in month_list], dtype=np.float32)
month_cos_values = np.cos(2 * np.pi * calendar_months / 12.0).astype(np.float32)
month_sin_values = np.sin(2 * np.pi * calendar_months / 12.0).astype(np.float32)

import gc

with torch.no_grad():
    for block_num, (r, c, h, w) in enumerate(tqdm(all_blocks, desc="Blocks")):
        # Initialise the GEE-band tensor with NaN; missing months stay NaN
        block_gee = np.full((T, n_gee_features, h, w), np.nan, dtype=np.float32)
        for m_idx in present_m_idx:
            block_gee[m_idx] = raster_handles[m_idx].read(
                window=Window(c, r, w, h))

        # Reshape from (T, C_gee, h, w) to (N=h*w, T, C_gee)
        N = h * w
        X_gee = block_gee.transpose(2, 3, 0, 1).reshape(N, T, n_gee_features)

        # Build the full 52-feature tensor
        X = np.full((N, T, n_features), np.nan, dtype=np.float32)

        # Copy the 46 GEE bands into their correct positions in the full tensor
        for src_idx, feat_name in enumerate(GEE_FEATURES):
            tgt_idx = config.MODEL_FEATURES.index(feat_name)
            X[:, :, tgt_idx] = X_gee[:, :, src_idx]

        # Compute delta features from the GEE-loaded bands (t=0 stays NaN)
        for band_name in config.DELTA_SOURCE_INDICES:
            src_idx = delta_source_indices[band_name]
            tgt_idx = delta_target_indices[f"delta_{band_name}"]
            band_values = X_gee[:, :, src_idx]
            X[:, 1:, tgt_idx] = band_values[:, :-1] - band_values[:, 1:]

        # Harmonic features - same value for every pixel at a given
        # calendar month, so broadcast across the (N, T) grid
        X[:, :, harmonic_target_indices["month_cos"]] = \
            month_cos_values[np.newaxis, :]
        X[:, :, harmonic_target_indices["month_sin"]] = \
            month_sin_values[np.newaxis, :]

        # Identify valid positions per (pixel, month) from the GEE bands
        # (delta at t=0 is always NaN; the model input mask handles it)
        nan_mask   = np.isnan(X_gee).any(axis=-1)
        valid_mask = ~nan_mask

        # Free the block_gee tensor now that we've moved the data into X
        del block_gee, X_gee, nan_mask

        any_valid_pixel = valid_mask.any(axis=1)
        eco_block       = eco_mask_30m[r:r+h, c:c+w].flatten()
        eligible        = any_valid_pixel & (eco_block == 1)

        out_sev  = np.full(N, np.nan, dtype=np.float32)
        out_per  = np.full(N, np.nan, dtype=np.float32)
        out_rec  = np.full(N, np.nan, dtype=np.float32)
        out_brn  = np.full(N, np.nan, dtype=np.float32)
        out_mrc  = np.full(N, np.nan, dtype=np.float32)   # model-derived recurrence

        if eligible.sum() > 0:
            X_elig    = X[eligible]
            mask_elig = valid_mask[eligible]

            # Free the full X now that we've extracted eligible pixels
            del X, valid_mask, any_valid_pixel

            # Replace NaN with 0 (masked in the model regardless)
            X_elig = np.nan_to_num(X_elig, nan=0.0)

            # Scale ONLY the spectral columns; LC one-hot stays at 0/1
            N_elig  = X_elig.shape[0]
            X_flat  = X_elig.reshape(-1, n_features)
            X_flat[:, spectral_cols_arr] = scaler.transform(
                X_flat[:, spectral_cols_arr])
            X_scaled = X_flat.reshape(N_elig, T, n_features).astype(np.float32)

            # Free the pre-scaled tensor
            del X_elig, X_flat

            sev_out = np.zeros(N_elig, dtype=np.float32)
            per_out = np.zeros(N_elig, dtype=np.float32)
            rec_out = np.zeros(N_elig, dtype=np.float32)
            brn_out = np.zeros(N_elig, dtype=np.float32)
            mrc_out = np.zeros(N_elig, dtype=np.float32)   # model_recurrence

            for i in range(0, N_elig, INFERENCE_BATCH):
                j = min(i + INFERENCE_BATCH, N_elig)
                Xb = torch.from_numpy(X_scaled[i:j]).to(DEVICE)
                Mb = torch.from_numpy(mask_elig[i:j]).to(DEVICE)
                out = model(Xb, Mb)
                sev_out[i:j] = out["severity"    ].cpu().numpy()
                per_out[i:j] = out["persistence" ].cpu().numpy()
                rec_out[i:j] = out["recovery_gap"].cpu().numpy()
                # Aggregate burned probability per pixel across months.
                # burned_probs shape: (batch, T=108).
                # - burned_prob raster: max over months (ever-burned indicator).
                # - model_recurrence raster: count of months above the
                #   F1-optimized threshold, normalized by 5 and capped at 1,
                #   matching the analytical recurrence target definition.
                BURN_THRESHOLD = 0.502
                burned_probs = torch.sigmoid(out["burned_logits"]).cpu().numpy()
                brn_out[i:j] = burned_probs.max(axis=1)
                mrc_out[i:j] = np.clip(
                    (burned_probs > BURN_THRESHOLD).sum(axis=1) / 5.0,
                    0.0, 1.0
                ).astype(np.float32)
                del Xb, Mb, out, burned_probs

            out_sev[eligible] = sev_out
            out_per[eligible] = per_out
            out_rec[eligible] = rec_out
            out_brn[eligible] = brn_out
            out_mrc[eligible] = mrc_out

            del X_scaled, mask_elig, sev_out, per_out, rec_out, brn_out, mrc_out
        else:
            del X, valid_mask, any_valid_pixel

        out_handles["severity"    ].write(out_sev.reshape(h, w), 1,
                                          window=Window(c, r, w, h))
        out_handles["persistence" ].write(out_per.reshape(h, w), 1,
                                          window=Window(c, r, w, h))
        out_handles["recovery_gap"].write(out_rec.reshape(h, w), 1,
                                          window=Window(c, r, w, h))
        out_handles["burned_prob" ].write(out_brn.reshape(h, w), 1,
                                          window=Window(c, r, w, h))
        out_handles["model_recurrence"].write(out_mrc.reshape(h, w), 1,
                                          window=Window(c, r, w, h))

        del out_sev, out_per, out_rec, out_brn, out_mrc, eligible, eco_block

        # Force garbage collection every 10 blocks
        if (block_num + 1) % 10 == 0:
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

for hh in raster_handles.values():
    hh.close()
for hh in out_handles.values():
    hh.close()

elapsed = time.time() - start_time
print(f"\nInference complete in {elapsed / 60:.1f} minutes")

Starting block-wise inference...


Blocks:   0%|          | 0/77 [00:00<?, ?it/s]


Inference complete in 23.0 minutes


## 7. Build the recurrence raster from the burn-count raster

In [19]:
with rasterio.open(ECO_MASK_10M) as src:
    burn_count_30m = src.read(
        1,                                # band 1 = burn count
        out_shape=(H, W),
        resampling=Resampling.average,
    )

recurrence_30m = np.minimum(burn_count_30m / 5.0, 1.0).astype(np.float32)
recurrence_30m[eco_mask_30m != 1] = np.nan

recurrence_path = OUT_DIR / "recurrence_30m.tif"
out_meta_rec = raster_meta.copy()
out_meta_rec.update(count=1, dtype="float32", nodata=np.nan)
with rasterio.open(recurrence_path, "w", **out_meta_rec) as dst:
    dst.write(recurrence_30m, 1)

print(f"Recurrence raster saved: {recurrence_path}")
print(f"  Mean (eligible pixels): {np.nanmean(recurrence_30m):.4f}")
print(f"  Max:                    {np.nanmax(recurrence_30m):.4f}")

Recurrence raster saved: /content/drive/MyDrive/thesisv4/outputs/inference_full_roi/recurrence_30m.tif
  Mean (eligible pixels): 0.0033
  Max:                    1.0000


## 8. Summary

### Model-derived recurrence

Alongside the four indicator rasters, this notebook writes `model_recurrence_30m.tif`,
derived from the classification head: for each pixel it counts the months in which
`burned_prob > 0.502`, then normalises by 5 and caps at 1.

It is the model-derived counterpart to `recurrence_30m.tif`, which is rasterised from
the fire-reference inventory. Both are kept so that model-detected and
reference-documented recurrence can be compared directly.

In [20]:
print("=" * 70)
print("Notebook 07b v3 complete.")
print("=" * 70)
print(f"\nOutput rasters in: {OUT_DIR}")
for f in sorted(OUT_DIR.glob("*.tif")):
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"  {f.name}  ({size_mb:.1f} MB)")
print("\nNext step: run Notebook 07c to upload these rasters to GEE.")

Notebook 07b v3 complete.

Output rasters in: /content/drive/MyDrive/thesisv4/outputs/inference_full_roi
  burned_prob_30m.tif  (16.1 MB)
  model_recurrence_30m.tif  (16.1 MB)
  persistence_30m.tif  (16.1 MB)
  recovery_gap_30m.tif  (16.1 MB)
  recurrence_30m.tif  (16.1 MB)
  severity_30m.tif  (16.1 MB)

Next step: run Notebook 07c to upload these rasters to GEE.
